# Learning From Data

## Workshop 3 orientation: investigating model behaviour

So far, the workshops have focused on the design levers in a learning problem: the data we collect, the functions we allow, and the procedure we use to select one trained solution.

This workshop uses the same frame for a different purpose. Instead of treating data, hypothesis space, and optimization as separate theory topics, we will use them as lenses for investigating a trained model.

By the end of this notebook, you should be able to:

1. explain how data, hypothesis space, and optimization shape a trained solution;
2. describe the shift from model design to model investigation;
3. identify the solar PV prediction task and the workshop sequence.

This opening notebook is only an orientation. The data exploration, training run, evaluation report, and group investigations begin in the notebooks that follow.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

workshop_pkg = importlib.import_module(PACKAGE_NAME)
package_version = getattr(workshop_pkg, "__version__", "unknown")
print(
    f"Workshop 3 environment ready. Package: {PACKAGE_NAME} {package_version}. "
    f"Repository: {repo_dir or 'installed environment'}"
)


<br>

## 1. Workshop 2 Recap

A learning system combines evidence, possible functions, and a training procedure. Workshop 2 studied these pieces as design choices.

$$
(\mathcal{D},\mathcal{H},\mathcal{O})\mapsto \widehat{h},
\qquad
\widehat{h}=h_{\widehat{\theta}},
\qquad
\widehat{\theta}\in
\operatorname*{arg\,min}_{\theta\in\Theta}
\widehat{R}_{\mathcal{D}}(h_\theta).
$$

The data determine what evidence is available. The hypothesis space determines what functions the model can represent. The optimizer and objective determine which available function is selected.

We consider a solution $\widehat{h} \in \mathcal{S}$ to be valid when

$$
\operatorname{RMSE}_{\mathrm{val}}(\widehat{h})\leq \varepsilon_{\mathrm{overall}},
\qquad
\operatorname{RMSE}_{\mathrm{val},X_{\mathrm{key}}}(\widehat{h})\leq \varepsilon_{\mathrm{key}}
$$

The same levers that shape learning can also help us investigate why a trained solution behaves the way it does.


<br>

## 2. From Design To Investigation

A single average score can tell us whether a model is acceptable, but it rarely tells us what to do next. When a trained model is not good enough, we need evidence about where it succeeds, where it struggles, and what kind of change would test a plausible explanation.

This is the core shift for Workshop 3:

> We have focused so far on the design levers for the data, hypothesis, and optimization spaces. Now we use those levers to probe model behaviour and diagnose the training process.


<br>

## 3. Solar PV Scenario

We will work with a synthetic but physically motivated solar PV regression problem.

The setting is simple: we observe operating conditions for a solar PV system and want to predict its normalized power output. Normalized power is a unitless output scaled relative to the system's expected output, so values are roughly interpretable on a 0-to-1 scale.

The example is synthetic so we can run controlled comparisons and inspect evidence clearly. The workflow mirrors a real modelling project: define the task, train a baseline, evaluate the baseline, investigate behaviour, and test interventions.


In [ ]:
# Load workshop helpers and prepare the shared data objects.
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt

bundle = pv.make_workshop3_bundle("baseline", seed=7)
project_data = pv.concat_data(
    [bundle["train"], bundle["validation"], bundle["test"]],
    name="project data",
)
input_columns = ["irradiance", "ambient_temperature", "tilt_angle"]
target_column = bundle["target"]

labels = {
    "irradiance": "irradiance (W/m2)",
    "ambient_temperature": "ambient temperature (C)",
    "tilt_angle": "tilt angle (degrees)",
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, column in zip(axes, input_columns):
    ax.scatter(
        project_data[column],
        project_data[target_column],
        s=18,
        alpha=0.42,
        color=pv.COLORS["train"],
        edgecolor="none",
    )
    ax.set_title(f"Power vs. {column}")
    ax.set_xlabel(labels[column])
    ax.grid(alpha=0.18)
axes[0].set_ylabel("normalized power")
fig.tight_layout()
plt.show()


<br>

## 4. Prediction Task

The prediction task is:

> Given measured operating conditions, predict normalized solar PV power output.

Let $P$ denote normalized power. Let $I$ be irradiance, $T_a$ be ambient temperature, and $\alpha$ be tilt angle. The input and target are:

$$
x=(I,T_a,\alpha)\in X \subset\mathbb{R}^3,
\qquad
P\in[0,1].
$$

The observed dataset is:

$$
\mathcal{D}=\{(x_i,P_i)\}_{i=1}^{n}.
$$

The modelling goal is to select a prediction rule:

$$
\widehat{h}\in\mathcal{H},
\qquad
\widehat{P}=\widehat{h}(x)\approx P,
\qquad
\widehat{h}: X \to [0,1].
$$

The project will call a trained solution acceptable only if validation error is small overall and inside the key operating range:

$$
X_{\mathrm{key}}
=
\left\{(I,T_a,\alpha):
I\geq700\,\mathrm{W}/\mathrm{m}^2,
\;T_a\geq30^{\circ}\mathrm{C},
\;15^{\circ}\leq\alpha\leq45^{\circ}
\right\}.
$$

In the next notebook, we will inspect the inputs, inspect $P$, and define the numeric thresholds for this acceptable-solution condition.


<br>

## 5. Workshop Roadmap

| Notebook | Role |
|---|---|
| `01_problem_setup.ipynb` | Define the PV prediction task and success criterion |
| `02_our_approach.ipynb` | Train a reasonable baseline MLP |
| `03_evaluation.ipynb` | Introduce the shared evaluation report and breakout activity |
| `03a_data_space_interventions.ipynb` | Run the data-space investigation |
| `03b_hypothesis_space_interventions.ipynb` | Run the hypothesis-space investigation |
| `03c_optimizer_space_interventions.ipynb` | Run the optimizer/objective investigation |
| `04_synthesis.ipynb` | Synthesize how to diagnose learning problems |
| `05_experiment_lab.ipynb` | Combine data, hypothesis, and optimizer/objective levers in a capstone challenge |

The capstone score matters less than the reasoning process used to choose, test, and interpret an intervention.


<br>

## 6. Activity Role

During the breakout activity, each group works through one intervention notebook. The job is to:

1. read the investigation brief;
2. inspect the evidence;
3. choose an intervention hypothesis;
4. run the provided experiments;
5. record the result and conclusion;
6. report back on the process.

The focus is scientific reasoning from evidence rather than implementation code.

For the report-back, be ready to complete this sentence:

```text
We investigated ______ because the evidence suggested ______. We changed ______. The result made us more/less confident because ______.
```


<br>

## 7. Handoff

The next notebook narrows this broad project into a concrete prediction task by showing exactly what data the learner receives and what success will mean.
